# Prerequisites
본 `ipynb` 은 `Python=3.12` 에서 작성하였습니다. Package dependency 를 해결하기 위해 아래 cell 을 실행해주세요.

## Install Python packages

In [ ]:
%pip -q install -U dotenv openai

## Load environment variables from a .env file
secret 노출을 피하고 notebook 들간의 일관된 환경변수를 설정하기 위해 `dotenv` 을 이용한다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

# Optimization
OpenAI 는 다양한 최적화 도구를 제공한다. 이를 통해 성능과 비용을 선택적으로 개선해 나갈 수 있다.

In [ ]:
from openai import OpenAI

# OpenAI 클라이언트를 생성합니다.
client = OpenAI(
    base_url=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
)

## Prompt Caching
Prompt caching 는 OpenAI server-side 에서 prompt 를 저장하는 기술로, 사용자는 이를 통해 LLM 모델에 요청의 latency 와 token usage 를 효과적으로 줄일 수 있다. 간단한 사용 방법을 알아보자.
- https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/prompt-caching

In [ ]:
from openai.types.chat import ChatCompletionUserMessageParam
import time

# chat completions API 호출
# - 10회 호출하여 각 호출마다 prompt tokens details를 출력합니다.
# - 첫 호출에서는 cached_tokens 이 0 이지만, 이후 호출에서는 cached_tokens이 증가하는 것을 확인할 수 있습니다.
for i in range(10):
    start_t = time.time()
    response = client.chat.completions.create(
        model=AZURE_OPENAI_CHAT_DEPLOYMENT,
        messages=[
            ChatCompletionUserMessageParam(
                role="user",
                content="입력되는 숫자에 이어서 100개를 더 출력해줘.\n" + "\n".join([f"{i}" for i in range(1000)]),
            ),
        ],
    )
    end_t = time.time()
    print(
        f"--- Response {i+1} --- "
        f"{response.usage.prompt_tokens_details.model_dump()} "
        f"(Elapsed time: {end_t - start_t:.2f} seconds)"
    )

위와 같이 default prompt caching 를 implicit prompt caching 라고 부른다. default 에서는 deployment 를 이루는 single serving-endpoint 에서 in-memory 에 prompt 가 caching 된다. in-memory cache 는 cache routing 이후 cache-hit 가 없다면 기본 5-6분 이내에 expired 되며 최대 1시간 유지된다.

다른 prompt caching 으로는 gpu-storage 에 저장하는 extended prompt cache 가 있다. extended 는 최대 24시간까지 retention 이 유지되고, `prompt_cache_retention=24h` 으로 설정한다.

In [ ]:
from openai.types.chat import ChatCompletionUserMessageParam
import time

# chat completions API 호출
# - 10회 호출하여 각 호출마다 prompt tokens details를 출력합니다.
# - 첫 호출에서는 cached_tokens 이 0 이지만, 이후 호출에서는 cached_tokens이 증가하는 것을 확인할 수 있습니다.
for i in range(10):
    start_t = time.time()
    response = client.chat.completions.create(
        model=AZURE_OPENAI_CHAT_DEPLOYMENT,
        prompt_cache_retention="24h",
        messages=[
            ChatCompletionUserMessageParam(
                role="user",
                content="내가 보여주는 입력되는 숫자에 이어서 100개를 더 출력해줘.\n" + "\n".join([f"{i}" for i in range(1000)]),
            ),
        ],
    )
    end_t = time.time()
    print(
        f"--- Response {i+1} --- "
        f"{response.usage.prompt_tokens_details.model_dump()} "
        f"(Elapsed time: {end_t - start_t:.2f} seconds)"
    )

prompt caching 이 되더라도 한번씩 `cached_tokens=0` 으로 되는 응답이 있다. Azure OpenAI 는 deployment cluster 내 다수의 serving endpoints 를 운영하는데, prompt caching 이 되는 storage 는 단일 serving endpoint 에 위치한다. 다수의 serving endpoints 에 걸처 caching 된 prompt 를 공유할 수 없는 구조이라서, cache-miss 가 가끔씩 발생한다.

cache-hit 를 최적화하기 위해 사용할 수 있는 argument 는 `prompt_cache_key` 이다. 사용방법은 아래와 같다.

In [ ]:
from openai.types.chat import ChatCompletionUserMessageParam
import time

# chat completions API 호출
# - 10회 호출하여 각 호출마다 prompt tokens details를 출력합니다.
# - 첫 호출에서는 cached_tokens 이 0 이지만, 이후 호출에서는 cached_tokens이 증가하는 것을 확인할 수 있습니다.
for i in range(10):
    start_t = time.time()
    response = client.chat.completions.create(
        model=AZURE_OPENAI_CHAT_DEPLOYMENT,
        prompt_cache_retention="24h",
        prompt_cache_key="prompt-cache-key-1",
        messages=[
            ChatCompletionUserMessageParam(
                role="user",
                content="내가 보여주는 입력되는 숫자에 이어서 100개를 더 출력해줘.\n" + "\n".join([f"{i}" for i in range(1000)]),
            ),
        ],
    )
    end_t = time.time()
    print(
        f"--- Response {i+1} --- "
        f"{response.usage.prompt_tokens_details.model_dump()} "
        f"(Elapsed time: {end_t - start_t:.2f} seconds)"
    )

# chat completions API 호출
# - prompt_cache_key를 변경하여 호출하면, 캐시가 적용되지 않는 것을 확인할 수 있습니다.
start_t = time.time()
response = client.chat.completions.create(
    model=AZURE_OPENAI_CHAT_DEPLOYMENT,
    prompt_cache_retention="24h",
    prompt_cache_key="prompt-cache-key-2",
    messages=[
        ChatCompletionUserMessageParam(
            role="user",
            content="내가 보여주는 입력되는 숫자에 이어서 100개를 더 출력해줘.\n" + "\n".join([f"{i}" for i in range(1000)]),
        ),
    ],
)
end_t = time.time()
print(
    f"--- Response with different prompt_cache_key --- "
    f"{response.usage.prompt_tokens_details.model_dump()} "
    f"(Elapsed time: {end_t - start_t:.2f} seconds)"
)

## Structed Outputs
출력되는 token 량은 LLM 모델이 non-deterministic 하기에 요청마다 다르다. Output Schema 를 정의하여 응답 format 으로 전달하면 해당 구조에 맞게 generate 한다. 모델에 따라 기대하는 token usage 가 달라질 수 있기에 이점 유념하도록 한다.

In [ ]:
from pydantic import BaseModel

class IntentEvent(BaseModel):
    intent: str

# LLM 모델인 gpt-4.1 을 사용하여 한국어로 된 문장의 의도를 분석하고, structed output 으로 반환받아보자.
# gpt-4.1 은 intent 에 해당하는 token 만 generate 한다.
response = client.beta.chat.completions.parse(
    model="gpt-4.1", # replace with the model deployment name of your gpt-4o 2024-08-06 deployment
    messages=[
        {"role": "system", "content": "너는 대화 의도 분석을 전문으로 하는 AI야."},
        {"role": "user", "content": "Alice와 Bob이 금요일에 과학 박람회에 가기로 했어."},
    ],
    response_format=IntentEvent,
)
print("---- gpt-4.1 ----")
print(f"응답: {response.choices[0].message.parsed}")
print(response.usage.model_dump_json(indent=2))

# LLM 모델인 gpt-5 을 사용하여 한국어로 된 문장의 의도를 분석하고, structed output 으로 반환받아보자.
# gpt-5 은 gpt-4.1 보다 많은 token 을 generate 하고, 후처리로 intent 를 추출한다.
response = client.beta.chat.completions.parse(
    model="gpt-5.1",
    messages=[
        {"role": "system", "content": "너는 대화 의도 분석을 전문으로 하는 AI야."},
        {"role": "user", "content": "Alice와 Bob이 금요일에 과학 박람회에 가기로 했어."},
    ],
    response_format=IntentEvent,
)
print("---- gpt-5 ----")
print(f"응답: {response.choices[0].message.parsed}")
print(response.usage.model_dump_json(indent=2))

# responses API 를 사용해보자. 동일하지만...
response = client.responses.parse(
    model="gpt-5.1",
    input=[
        {"role": "system", "content": "너는 대화 의도 분석을 전문으로 하는 AI야."},
        {"role": "user", "content": "Alice와 Bob이 금요일에 과학 박람회에 가기로 했어."},
    ],
    text_format=IntentEvent,
)
print("---- gpt-5 ----")
print(f"응답: {response.output_text}")
print(response.usage.model_dump_json(indent=2))